In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


In [14]:
df = pd.read_csv("Iris.csv")
X = df.drop(columns=["Species"])

# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [15]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans.fit_predict(X_scaled)

sil_kmeans = silhouette_score(X_scaled, labels_kmeans)
print("K-Means Silhouette Score:", sil_kmeans)


K-Means Silhouette Score: 0.452949780355554


In [16]:
from sklearn.metrics import pairwise_distances

def k_medoids(X, k, max_iter=100):
    n = X.shape[0]
    medoids = np.random.choice(n, k, replace=False)

    for _ in range(max_iter):
        distances = pairwise_distances(X, X[medoids])
        labels = np.argmin(distances, axis=1)

        new_medoids = []
        for i in range(k):
            cluster_points = np.where(labels == i)[0]
            cluster_distances = pairwise_distances(X[cluster_points])
            new_medoids.append(cluster_points[np.argmin(cluster_distances.sum(axis=1))])

        new_medoids = np.array(new_medoids)
        if np.all(medoids == new_medoids):
            break
        medoids = new_medoids

    return labels

labels_kmedoids = k_medoids(X_scaled, k=3)

sil_kmedoids = silhouette_score(X_scaled, labels_kmedoids)

print("K-Medoids Silhouette Score:", sil_kmedoids)


K-Medoids Silhouette Score: 0.4511665527107823


In [22]:
from kmodes.kmodes import KModes

data_cat = pd.DataFrame({
    'Weather': ['Hot','Hot','Cold','Mild','Cold','Mild'],
    'Humidity': ['High','High','Low','Low','Low','High'],
    'Wind': ['Weak','Strong','Weak','Strong','Weak','Strong']
})

kmodes = KModes(n_clusters=2, init='Cao', random_state=42)
labels_kmodes = kmodes.fit_predict(data_cat)

print("K-Modes Cluster Labels:", labels_kmodes)


K-Modes Cluster Labels: [0 1 0 1 0 1]


In [23]:
from sklearn.cluster import AgglomerativeClustering

agnes_min = AgglomerativeClustering(
    n_clusters=3,
    linkage='single'
)
labels_agnes_min = agnes_min.fit_predict(X_scaled)

sil_agnes_min = silhouette_score(X_scaled, labels_agnes_min)
print("AGNES (Min) Silhouette:", sil_agnes_min)


AGNES (Min) Silhouette: 0.4784399371880524


In [19]:
agnes_max = AgglomerativeClustering(
    n_clusters=3,
    linkage='complete'
)
labels_agnes_max = agnes_max.fit_predict(X_scaled)

sil_agnes_max = silhouette_score(X_scaled, labels_agnes_max)
print("AGNES (Max) Silhouette:", sil_agnes_max)


AGNES (Max) Silhouette: 0.40583021522665685


In [21]:

from sklearn.cluster import KMeans
import numpy as np

def compute_sse(cluster):
    if len(cluster) == 0:
        return 0
    centroid = np.mean(cluster, axis=0)
    return np.sum((cluster - centroid) ** 2)


def diana_bisecting_kmeans(X, k):
    clusters = [np.arange(len(X))]

    while len(clusters) < k:

        sse_values = []
        for c in clusters:
            sse_values.append(compute_sse(X[c]))

        split_index = np.argmax(sse_values)
        cluster_to_split = clusters.pop(split_index)

        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
        split_labels = kmeans.fit_predict(X[cluster_to_split])

        cluster1 = cluster_to_split[split_labels == 0]
        cluster2 = cluster_to_split[split_labels == 1]

        clusters.append(cluster1)
        clusters.append(cluster2)

    labels = np.zeros(len(X), dtype=int)
    for i, c in enumerate(clusters):
        labels[c] = i

    return labels


labels_diana = diana_bisecting_kmeans(X_scaled, k=3)

sil_diana = silhouette_score(X_scaled, labels_diana)
print("DIANA (Bisecting K-Means) Silhouette:", sil_diana)


DIANA (Bisecting K-Means) Silhouette: 0.452949780355554
